# Week 6, Day 4: Linear dynamics: stability, oscillation and the line attractor

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w006/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Euler simulation of a rate network

Write `simulate(W, x0, tau, dt, n_steps, u=None)` using Euler's method for $\tau\dot{\mathbf{x}} = -\mathbf{x} + W\mathbf{x} + \mathbf{u}$. `u`, if given, is an array with one row of input per step. Return an array whose row k is the state after k steps (row 0 is `x0`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate(W, x0, tau, dt, n_steps, u=None):
    X = np.zeros((n_steps + 1, len(x0)))
    X[0] = x0
    for k in range(n_steps):
        inp = 0 if u is None else u[k]
        ...
    return X

X = simulate(np.array([[0.0, -1.2], [1.2, 0.0]]) * 0.9 + 0.5 * np.eye(2), np.array([1.0, 0.0]), tau=0.01, dt=0.0002, n_steps=1000)
plt.figure(figsize=(5, 3)); plt.plot(np.arange(1001) * 0.0002 * 1000, X); plt.xlabel("time (ms)"); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "returns n_steps + 1 rows, one column per unit"
    X = simulate(np.zeros((3, 3)), np.ones(3), tau=0.1, dt=0.001, n_steps=50)
    assert X.shape == (51, 3) and np.allclose(X[0], 1)

def test_decay():
    "with W diagonal, each unit decays with time constant tau / (1 - w)"
    W = np.diag([0.0, 0.5])
    X = simulate(W, np.array([1.0, 1.0]), tau=0.1, dt=0.0005, n_steps=400)
    t = 400 * 0.0005
    assert np.allclose(X[-1], np.exp(-(1 - np.diag(W)) * t / 0.1), rtol=0.01)

def test_input():
    "a constant input drives an isolated unit to the steady state u / (1 - w)"
    X = simulate(np.array([[0.5]]), np.array([0.0]), tau=0.01, dt=0.0005, n_steps=2000, u=np.full((2000, 1), 1.0))
    assert np.isclose(X[-1, 0], 2.0, rtol=0.01)

test_shape()
test_decay()
test_input()
print("All checks passed")

In [ ]:
Wosc = np.array([[0.0, -1.2], [1.2, 0.0]]) * 0.9 + 0.5 * np.eye(2)
lam = np.linalg.eigvals(Wosc)
tau = 0.01
print("eigenvalues of W:", lam.round(3))
print("decay time constant:", round(1000 * tau / (1 - lam[0].real), 1), "ms")
print("predicted frequency:", round(abs(lam[0].imag) / (2 * np.pi * tau), 1), "Hz")

### Exercise: Tune the integrator

With τ = 0.1 s and a = 0.5, write `tau_eff(lam, tau)` and choose `b_tuned` so that the sum mode lasts 20 s. Simulate four saccade commands, read eye position as the mean of the two neurons, and measure the fractional drift over the last 2 s: `drift = (pos at 4 s minus pos at 6 s) / pos at 4 s`. Repeat with $a + b$ lowered by 0.02.

In [ ]:
tau, a, dt = 0.1, 0.5, 0.001
def tau_eff(lam, tau):
    return ...

b_tuned = ...

n = 6000                                  # 6 s
u = np.zeros((n, 2))
for t_sac, size in [(0.5, 3.0), (1.5, 2.0), (2.5, -4.0), (3.5, 2.5)]:
    k = int(t_sac / dt); u[k:k + 20] = size / (20 * dt) * tau   # brief burst into both neurons

def run(b):
    X = simulate(np.array([[a, b], [b, a]]), np.zeros(2), tau, dt, n, u)
    return X.mean(axis=1)

pos_tuned, pos_mis = run(b_tuned), run(b_tuned - 0.02)
drift_tuned = ...
drift_mistuned = ...

t = np.arange(n + 1) * dt
plt.figure(figsize=(6, 3)); plt.plot(t, pos_tuned, label="tuned (a + b = 0.995)"); plt.plot(t, pos_mis, label="mistuned by 0.02")
plt.xlabel("time (s)"); plt.ylabel("eye position (a.u.)"); plt.legend(frameon=False); plt.show()
print("drift tuned:", round(drift_tuned, 3), " mistuned:", round(drift_mistuned, 3))

In [ ]:
import numpy as np
def test_tau_eff():
    "tau_eff gives tau / (1 - lambda)"
    assert np.isclose(tau_eff(0.9, 0.1), 1.0) and np.isclose(tau_eff(0.995, 0.1), 20.0)

def test_b():
    "b_tuned makes the sum mode's time constant 20 s"
    assert np.isclose(tau_eff(a + b_tuned, tau), 20.0)

def test_hold():
    "the tuned network holds eye position: about 10% drift over 2 s, as a 20 s time constant predicts"
    assert 0.07 < drift_tuned < 0.12, f"drift {drift_tuned:.3f}"

def test_mistuned():
    "a 2% mistuning makes the position decay noticeably"
    assert drift_mistuned > 0.2

test_tau_eff()
test_b()
test_hold()
test_mistuned()
print("All checks passed")